# Box planner in simulation

Plan a sequence of pivots and slides that brings a box from a start pose to a goal pose, execute it with the
executor in kinematic simulation, and replay it in Swift. Same code as `experiments/run_plan.py`; the tasks and
all parameters are in `experiments/configs/box_plan.yaml`.

Kernel: the repository's `.venv` (`uv sync --extra sim --extra robot --extra dev`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

here = Path.cwd() if (Path.cwd() / "run_plan.py").exists() else Path.cwd() / "experiments"
sys.path.insert(0, str(here))
import run_plan as rp

# Ablation switches: free_rotation is the execution model (True: the box may rotate about the finger axis during
# a pivot; False: rigid grasp). plan_free_rotation is the model of the planner's feasibility check (None: the same).
cfg = rp.load_config(here / "configs" / "box_plan.yaml", free_rotation=True, plan_free_rotation=None)
robot = rp.make_robot(cfg)
planner = rp.make_planner(cfg, robot)
list(cfg["tasks"])

## Plan and execute one task

`run_task` plans, then runs the plan on the simulated arm at 1 kHz and checks the result: every segment completed,
the QP never infeasible, joint speeds within the limits, object at the goal (up to the box symmetry), arm back at
the ready pose.

In [ ]:
res = rp.run_task(cfg, robot, "slide_pivot", planner)
rp.report(res)

In [ ]:
for i, s in enumerate(res["plan"].steps):
    p = s.T_after[:3, 3]
    print(f"step {i}: {s.kind:5s} gripper width {s.plan.width * 1e3:.0f} mm, box centre after: {np.round(p, 3)}")

## Replay in Swift

Opens a browser tab on the machine that runs the kernel (blue: the box, green: the goal pose).
`rp.show(path, notebook=True)` draws inside the cell instead; `speed=2` plays twice as fast.

In [ ]:
path = rp.save(res, cfg)
env = rp.show(path)

## Joint speeds and bearing angle

In [ ]:
from pandaSim.control.contact_qp import FER_QD_MAX

a = res["log"].arrays()
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(a["t"], np.abs(a["qd"][:, :7]) / FER_QD_MAX)
ax[0].axhline(0.8, color="k", ls="--", lw=0.8)
ax[0].set_ylabel("|joint speed| / limit")
ax[1].plot(a["t"], np.degrees(a["phi"]))
ax[1].set_ylabel("phi (model) [deg]")
ax[1].set_xlabel("t [s]")
for s in res["log"].segments:
    for x in ax:
        x.axvline(s["t0"], color="0.8", lw=0.6)
    ax[0].text(s["t0"], 0.82, s["name"], fontsize=7, rotation=90, va="bottom")
plt.tight_layout()

## All tasks, free rotation against rigid grasp

Slides are executed with the rigid model in both cases, so the switch only changes pivots.

In [ ]:
rows = {}
for free in (True, False):
    c = rp.load_config(here / "configs" / "box_plan.yaml", free_rotation=free)
    pl = rp.make_planner(c, robot)
    for name in c["tasks"]:
        r = rp.run_task(c, robot, name, pl)
        kinds = " -> ".join(s.kind for s in r["plan"].steps) if r["plan"].found else "no plan"
        rows.setdefault(name, []).append(f"{kinds} ({'pass' if rp.passed(r) else 'fail'})")
print(f"{'task':16s} {'free rotation':28s} rigid")
for name, (f, r) in rows.items():
    print(f"{name:16s} {f:28s} {r}")

## Your own task

Poses are the box centre `xy` on the table and roll-pitch-yaw in degrees; the height follows from the face that is
down. The gripper only spans the 0.06 m side, so the box cannot be grasped while it lies on its largest face.

In [ ]:
cfg["tasks"]["mine"] = {"start": {"xy": [0.50, 0.00], "rpy": [0, -90, 0]},
                        "goal":  {"xy": [0.50, 0.10], "rpy": [0, 0, 30]}}
mine = rp.run_task(cfg, robot, "mine", planner)
rp.report(mine)